# Notebook 3 — Clasificación multiclase y regresión con redes neuronales

**Curso:** Deep Learning  
**Framework:** TensorFlow/Keras  
**Docente:** Jersson  
**Estudiantes:** Juan David Tejedor Medina y Miguel Guerardo Moreno Aveldaño  
**Entrega:** Actividades desarrolladas desde la sección 25

Este notebook desarrolla dos aplicaciones:

1. Clasificación multiclase con **8 características de entrada** y **4 clases de salida**.
2. Regresión con evaluación mediante **MAE, MSE, RMSE, \(R^2\) y MAPE**.

Al final se incluyen los experimentos, las preguntas de reflexión, la síntesis y el reto final completamente desarrollados.


### Edición para portafolio

Trabajo académico recuperado de la especialización. Se conservan el código, las explicaciones y las atribuciones originales. Se retiraron las salidas y los metadatos de ejecución para facilitar su lectura y revisión. Las conclusiones conservadas pertenecen a la entrega original; los entrenamientos de Deep Learning no se repitieron al organizar este repositorio. Ver [procedencia y autoría](../../docs/PROCEDENCIA.md).


## Objetivos de aprendizaje

- Construir una red neuronal para clasificación multiclase.
- Interpretar una salida Softmax.
- Evaluar un clasificador con accuracy, matriz de confusión, precision, recall y F1.
- Construir una red neuronal para regresión.
- Calcular e interpretar MAE, MSE, RMSE, \(R^2\) y MAPE.
- Analizar valores predichos, residuos y una línea base.


## 1. Preparación del entorno

1.   Elemento de la lista
2.   Elemento de la lista



In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    recall_score,
    mean_absolute_percentage_error,
)

import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


# Parte I — Clasificación multiclase

La matriz de entrada tendrá ocho características:

\[
X\in\mathbb{R}^{m\times 8}
\]

La variable objetivo contiene cuatro clases:

\[
y\in\{0,1,2,3\}
\]

La red producirá cuatro probabilidades por observación.


## 2. Generación del conjunto de datos

In [ ]:
X_class, y_class = make_classification(
    n_samples=2400,
    n_features=8,
    n_informative=6,
    n_redundant=2,
    n_classes=4,
    n_clusters_per_class=1,
    class_sep=1.5,
    flip_y=0.03,
    weights=[0.25, 0.25, 0.25, 0.25],
    random_state=SEED,
)

print("Forma de X:", X_class.shape)
print("Forma de y:", y_class.shape)
print("Clases:", np.unique(y_class))


In [ ]:
feature_names = [f"x{i}" for i in range(1, 9)]
classification_df = pd.DataFrame(X_class, columns=feature_names)
classification_df["clase"] = y_class
classification_df.head()


In [ ]:
classification_df["clase"].value_counts().sort_index()


## 3. Exploración visual

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_class[:, 0], X_class[:, 1], c=y_class, alpha=0.65)
plt.xlabel("Característica x1")
plt.ylabel("Característica x2")
plt.title("Proyección del problema multiclase")
plt.grid(alpha=0.25)
plt.show()


La gráfica solo muestra dos de las ocho dimensiones. La separación real depende del espacio completo de características.

## 4. División y estandarización

In [ ]:
X_train_full_c, X_test_c, y_train_full_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.20, stratify=y_class, random_state=SEED
)

X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(
    X_train_full_c, y_train_full_c,
    test_size=0.20, stratify=y_train_full_c, random_state=SEED
)

scaler_class = StandardScaler()
X_train_c_scaled = scaler_class.fit_transform(X_train_c)
X_val_c_scaled = scaler_class.transform(X_val_c)
X_test_c_scaled = scaler_class.transform(X_test_c)

print("Entrenamiento:", X_train_c_scaled.shape)
print("Validación:", X_val_c_scaled.shape)
print("Prueba:", X_test_c_scaled.shape)


## 5. Softmax

Para cuatro clases, Softmax calcula:

\[
p_k=\frac{e^{z_k}}{\sum_{j=1}^{4}e^{z_j}}
\]

y garantiza:

\[
\sum_{k=1}^{4}p_k=1
\]

La clase predicha es:

\[
\hat y=\arg\max_k p_k
\]


In [ ]:
logits = tf.constant([[2.0, 1.0, 0.5, -0.5]])
probs = tf.nn.softmax(logits)

print("Probabilidades:", probs.numpy())
print("Suma:", probs.numpy().sum())
print("Clase predicha:", np.argmax(probs.numpy(), axis=1))


## 6. Arquitectura del clasificador

\[
8\rightarrow 32\rightarrow 16\rightarrow 4
\]

La salida tiene cuatro neuronas con activación Softmax.


In [ ]:
classifier = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8,), name="entrada"),
    tf.keras.layers.Dense(32, activation="relu", name="oculta_1"),
    tf.keras.layers.Dense(16, activation="relu", name="oculta_2"),
    tf.keras.layers.Dense(4, activation="softmax", name="salida"),
], name="clasificador_multiclase")

classifier.summary()


## 7. Compilación

Como las etiquetas están codificadas con enteros 0, 1, 2 y 3, se utiliza `SparseCategoricalCrossentropy`.


In [ ]:
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)


## 8. Entrenamiento

In [ ]:
classification_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6
    ),
]

history_class = classifier.fit(
    X_train_c_scaled,
    y_train_c,
    validation_data=(X_val_c_scaled, y_val_c),
    epochs=250,
    batch_size=32,
    callbacks=classification_callbacks,
    verbose=0,
)

print("Épocas ejecutadas:", len(history_class.history["loss"]))


## 9. Curvas de aprendizaje

In [ ]:
history_class_df = pd.DataFrame(history_class.history)

plt.figure(figsize=(8, 5))
plt.plot(history_class_df["loss"], label="Entrenamiento")
plt.plot(history_class_df["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Clasificación multiclase: pérdida")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_class_df["accuracy"], label="Entrenamiento")
plt.plot(history_class_df["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Exactitud")
plt.title("Clasificación multiclase: exactitud")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 10. Evaluación del clasificador

In [ ]:
class_test_loss, class_test_accuracy = classifier.evaluate(
    X_test_c_scaled, y_test_c, verbose=0
)

print(f"Pérdida de prueba: {class_test_loss:.4f}")
print(f"Exactitud de prueba: {class_test_accuracy:.4f}")


In [ ]:
class_probabilities = classifier.predict(X_test_c_scaled, verbose=0)
class_predictions = np.argmax(class_probabilities, axis=1)

prediction_table = pd.DataFrame(
    class_probabilities[:10],
    columns=["P(clase 0)", "P(clase 1)", "P(clase 2)", "P(clase 3)"],
)
prediction_table["clase_real"] = y_test_c[:10]
prediction_table["clase_predicha"] = class_predictions[:10]
prediction_table


In [ ]:
print(classification_report(
    y_test_c,
    class_predictions,
    target_names=["Clase 0", "Clase 1", "Clase 2", "Clase 3"],
    digits=4,
))


## 11. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test_c, class_predictions)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Clase 0", "Clase 1", "Clase 2", "Clase 3"],
)
display.plot()
plt.title("Matriz de confusión")
plt.show()


### Interpretación

- La diagonal principal contiene predicciones correctas.
- Los valores fuera de la diagonal representan confusiones.
- Precision, recall y F1 permiten evaluar cada clase individualmente.


## 12. Actividad de clasificación

Modifique una variable a la vez:

- número de neuronas;
- número de capas;
- tasa de aprendizaje;
- función de activación;
- separación entre clases;
- proporción de etiquetas con ruido.

Registre accuracy, macro F1 y la clase con mayor dificultad.


In [ ]:
# Espacio para un clasificador alternativo.

# classifier_experiment = tf.keras.Sequential([
#     ...
# ])


# Parte II — Regresión

En regresión la salida es un valor continuo:

\[
\hat y\in\mathbb{R}
\]

Se utilizará el conjunto de datos de diabetes de scikit-learn, que contiene 10 características y una variable objetivo continua.


## 13. Carga del conjunto de regresión

In [ ]:
diabetes = load_diabetes()
X_reg = diabetes.data
y_reg = diabetes.target

print("Forma de X:", X_reg.shape)
print("Forma de y:", y_reg.shape)
print("Características:", diabetes.feature_names)


In [ ]:
regression_df = pd.DataFrame(X_reg, columns=diabetes.feature_names)
regression_df["objetivo"] = y_reg
regression_df.head()


## 14. Distribución de la salida

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(y_reg, bins=25, edgecolor="black")
plt.xlabel("Variable objetivo")
plt.ylabel("Frecuencia")
plt.title("Distribución de la variable objetivo")
plt.grid(alpha=0.20)
plt.show()


## 15. División y estandarización

In [ ]:
X_train_full_r, X_test_r, y_train_full_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=SEED
)

X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(
    X_train_full_r, y_train_full_r, test_size=0.20, random_state=SEED
)

scaler_reg = StandardScaler()
X_train_r_scaled = scaler_reg.fit_transform(X_train_r)
X_val_r_scaled = scaler_reg.transform(X_val_r)
X_test_r_scaled = scaler_reg.transform(X_test_r)

print("Entrenamiento:", X_train_r_scaled.shape)
print("Validación:", X_val_r_scaled.shape)
print("Prueba:", X_test_r_scaled.shape)


## 16. Arquitectura de regresión

\[
10\rightarrow 64\rightarrow 32\rightarrow 16\rightarrow 1
\]

La neurona de salida usa activación lineal porque debe producir un valor continuo.


In [ ]:
regressor = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10,), name="entrada"),
    tf.keras.layers.Dense(64, activation="relu", name="oculta_1"),
    tf.keras.layers.Dense(32, activation="relu", name="oculta_2"),
    tf.keras.layers.Dense(16, activation="relu", name="oculta_3"),
    tf.keras.layers.Dense(1, activation="linear", name="salida"),
], name="red_regresion")

regressor.summary()


## 17. Métricas de error

\[
MAE=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|
\]

\[
MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2
\]

\[
RMSE=\sqrt{MSE}
\]

\[
R^2=1-\frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}
\]


In [ ]:
regressor.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
        tf.keras.metrics.RootMeanSquaredError(name="rmse"),
    ],
)


## 18. Entrenamiento del regresor

In [ ]:
regression_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=30, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6
    ),
]

history_reg = regressor.fit(
    X_train_r_scaled,
    y_train_r,
    validation_data=(X_val_r_scaled, y_val_r),
    epochs=500,
    batch_size=32,
    callbacks=regression_callbacks,
    verbose=0,
)

print("Épocas ejecutadas:", len(history_reg.history["loss"]))


## 19. Curvas de regresión

In [ ]:
history_reg_df = pd.DataFrame(history_reg.history)

plt.figure(figsize=(8, 5))
plt.plot(history_reg_df["loss"], label="Entrenamiento")
plt.plot(history_reg_df["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Regresión: evolución de la pérdida")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_reg_df["mae"], label="MAE entrenamiento")
plt.plot(history_reg_df["val_mae"], label="MAE validación")
plt.xlabel("Época")
plt.ylabel("MAE")
plt.title("Regresión: evolución del MAE")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 20. Predicciones y métricas

In [ ]:
y_pred_r = regressor.predict(X_test_r_scaled, verbose=0).ravel()

mae = mean_absolute_error(y_test_r, y_pred_r)
mse = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_r, y_pred_r)
mape = mean_absolute_percentage_error(y_test_r, y_pred_r) * 100

regression_metrics = pd.DataFrame({
    "Métrica": ["MAE", "MSE", "RMSE", "R²", "MAPE (%)"],
    "Valor": [mae, mse, rmse, r2, mape],
})

regression_metrics


## 21. Valores reales y predichos

In [ ]:
comparison_regression = pd.DataFrame({
    "real": y_test_r,
    "predicho": y_pred_r,
})
comparison_regression["error"] = (
    comparison_regression["real"] - comparison_regression["predicho"]
)
comparison_regression.head(10)


In [ ]:
min_value = min(y_test_r.min(), y_pred_r.min())
max_value = max(y_test_r.max(), y_pred_r.max())

plt.figure(figsize=(7, 7))
plt.scatter(y_test_r, y_pred_r, alpha=0.75)
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Valores reales frente a predicciones")
plt.grid(alpha=0.25)
plt.show()


## 22. Análisis de residuos

El residuo es:

\[
e_i=y_i-\hat y_i
\]

Idealmente, los residuos deben distribuirse alrededor de cero sin patrones sistemáticos.


In [ ]:
residuals = y_test_r - y_pred_r

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_r, residuals, alpha=0.75)
plt.axhline(0, linestyle="--")
plt.xlabel("Valor predicho")
plt.ylabel("Residuo")
plt.title("Residuos frente a valores predichos")
plt.grid(alpha=0.25)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=20, edgecolor="black")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.title("Distribución de residuos")
plt.grid(alpha=0.20)
plt.show()


## 23. Comparación con una línea base

La línea base predice la media de los valores de entrenamiento para todas las observaciones.


In [ ]:
baseline_prediction = np.full_like(
    y_test_r,
    fill_value=np.mean(y_train_r),
    dtype=float,
)

baseline_mae = mean_absolute_error(y_test_r, baseline_prediction)
baseline_rmse = np.sqrt(mean_squared_error(y_test_r, baseline_prediction))
baseline_r2 = r2_score(y_test_r, baseline_prediction)

baseline_comparison = pd.DataFrame([
    {
        "Modelo": "Línea base: media",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse,
        "R²": baseline_r2,
    },
    {
        "Modelo": "Red neuronal",
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
    },
])

baseline_comparison


## 24. Comparación conceptual

In [ ]:
comparison_table = pd.DataFrame({
    "Elemento": [
        "Tipo de salida",
        "Neuronas de salida",
        "Activación final",
        "Función de pérdida",
        "Interpretación",
        "Métricas",
    ],
    "Clasificación multiclase": [
        "Categoría",
        "Una por clase",
        "Softmax",
        "Sparse categorical crossentropy",
        "Clase con mayor probabilidad",
        "Accuracy, precision, recall, F1",
    ],
    "Regresión": [
        "Valor continuo",
        "Una",
        "Lineal",
        "MSE o MAE",
        "Valor numérico",
        "MAE, RMSE, R², MAPE",
    ],
})

comparison_table


## 25. Actividad experimental — desarrollo completo

### Metodología

Se toma como control el clasificador `8 → 32 → 16 → 4`, con activación ReLU, `learning_rate=0.001`, separación entre clases de 1.5 y ruido de etiquetas de 0.03. En cada comparación se modifica **una sola variable** y las demás permanecen fijas.

Para regresión se comparan exactamente las tres arquitecturas solicitadas. Todos los modelos usan la misma partición de datos, estandarización, semilla, tamaño de lote y parada temprana. Esto hace que la comparación sea justa y reproducible.


### 25.1 Funciones auxiliares

Estas funciones construyen, entrenan y evalúan los modelos. La métrica **macro-F1** asigna la misma importancia a cada clase; la clase más difícil se identifica mediante el menor *recall*.


In [ ]:
def build_classifier(input_dim=8, hidden_layers=(32, 16), learning_rate=0.001,
                     l2_value=0.0, dropout_rate=0.0, seed=SEED):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    regularizer = tf.keras.regularizers.l2(l2_value) if l2_value > 0 else None

    layers = [tf.keras.layers.Input(shape=(input_dim,))]
    for units in hidden_layers:
        layers.append(tf.keras.layers.Dense(
            units, activation="relu", kernel_regularizer=regularizer
        ))
        if dropout_rate > 0:
            layers.append(tf.keras.layers.Dropout(dropout_rate))
    layers.append(tf.keras.layers.Dense(4, activation="softmax"))

    model = tf.keras.Sequential(layers)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def run_classification_experiment(name, group, hidden_layers=(32, 16),
                                  learning_rate=0.001, class_sep=1.5,
                                  flip_y=0.03, seed=SEED):
    X_exp, y_exp = make_classification(
        n_samples=2400,
        n_features=8,
        n_informative=6,
        n_redundant=2,
        n_classes=4,
        n_clusters_per_class=1,
        class_sep=class_sep,
        flip_y=flip_y,
        weights=[0.25, 0.25, 0.25, 0.25],
        random_state=seed,
    )
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X_exp, y_exp, test_size=0.20, stratify=y_exp, random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.20,
        stratify=y_train_full, random_state=seed
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    model = build_classifier(
        input_dim=8,
        hidden_layers=hidden_layers,
        learning_rate=learning_rate,
        seed=seed,
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6
        ),
    ]
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=180,
        batch_size=32,
        callbacks=callbacks,
        verbose=0,
    )
    probabilities = model.predict(X_test, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    recalls = recall_score(y_test, predictions, average=None, labels=[0, 1, 2, 3])

    result = {
        "Grupo": group,
        "Experimento": name,
        "Arquitectura": "8 → " + " → ".join(map(str, hidden_layers)) + " → 4",
        "Learning rate": learning_rate,
        "Ruido": flip_y,
        "Separación": class_sep,
        "Parámetros": model.count_params(),
        "Épocas": len(history.history["loss"]),
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(y_test, predictions, average="macro"),
        "Clase más difícil": int(np.argmin(recalls)),
        "Recall mínimo": float(np.min(recalls)),
    }
    return result, history.history


def build_regressor(hidden_layers, learning_rate=0.001, l2_value=0.0,
                    dropout_rate=0.0, seed=SEED):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    regularizer = tf.keras.regularizers.l2(l2_value) if l2_value > 0 else None

    layers = [tf.keras.layers.Input(shape=(10,))]
    for units in hidden_layers:
        layers.append(tf.keras.layers.Dense(
            units, activation="relu", kernel_regularizer=regularizer
        ))
        if dropout_rate > 0:
            layers.append(tf.keras.layers.Dropout(dropout_rate))
    layers.append(tf.keras.layers.Dense(1, activation="linear"))

    model = tf.keras.Sequential(layers)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


def train_and_evaluate_regressor(name, hidden_layers, l2_value=0.0,
                                 dropout_rate=0.0, seed=SEED):
    model = build_regressor(
        hidden_layers=hidden_layers,
        l2_value=l2_value,
        dropout_rate=dropout_rate,
        seed=seed,
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=30, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6
        ),
    ]
    history = model.fit(
        X_train_r_scaled, y_train_r,
        validation_data=(X_val_r_scaled, y_val_r),
        epochs=400,
        batch_size=32,
        callbacks=callbacks,
        verbose=0,
    )
    predictions = model.predict(X_test_r_scaled, verbose=0).ravel()
    mse_value = mean_squared_error(y_test_r, predictions)
    result = {
        "Modelo": name,
        "Arquitectura": "10 → " + " → ".join(map(str, hidden_layers)) + " → 1",
        "Parámetros": model.count_params(),
        "Épocas": len(history.history["loss"]),
        "MAE": mean_absolute_error(y_test_r, predictions),
        "RMSE": np.sqrt(mse_value),
        "R²": r2_score(y_test_r, predictions),
    }
    return result, predictions, history.history


print("Funciones auxiliares definidas correctamente.")

### 25.2 Experimentos de clasificación

Se ejecutan nueve configuraciones: el control y dos alternativas para cada variable solicitada. En los grupos de arquitectura, tasa de aprendizaje, ruido y separación, solo cambia la variable indicada.


In [ ]:
classification_configs = [
    {"name": "Control", "group": "Control"},
    {"name": "Arquitectura pequeña", "group": "Arquitectura", "hidden_layers": (16,)},
    {"name": "Arquitectura amplia", "group": "Arquitectura", "hidden_layers": (64, 32)},
    {"name": "LR = 0.01", "group": "Tasa de aprendizaje", "learning_rate": 0.01},
    {"name": "LR = 0.0001", "group": "Tasa de aprendizaje", "learning_rate": 0.0001},
    {"name": "Sin ruido", "group": "Ruido", "flip_y": 0.00},
    {"name": "Ruido = 0.12", "group": "Ruido", "flip_y": 0.12},
    {"name": "Separación = 0.8", "group": "Separación", "class_sep": 0.8},
    {"name": "Separación = 2.2", "group": "Separación", "class_sep": 2.2},
]

classification_experiments = []
classification_histories = {}

for config in classification_configs:
    params = {
        "hidden_layers": (32, 16),
        "learning_rate": 0.001,
        "class_sep": 1.5,
        "flip_y": 0.03,
    }
    params.update({k: v for k, v in config.items() if k not in {"name", "group"}})
    result, history = run_classification_experiment(
        config["name"], config["group"], **params
    )
    classification_experiments.append(result)
    classification_histories[config["name"]] = history
    print(f"Completado: {config['name']} ({result['Épocas']} épocas)")

classification_results = pd.DataFrame(classification_experiments)
classification_results.round({
    "Learning rate": 4, "Ruido": 2, "Separación": 1,
    "Accuracy": 4, "Macro F1": 4, "Recall mínimo": 4,
})

In [ ]:
plot_data = classification_results.sort_values("Accuracy")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(plot_data["Experimento"], plot_data["Accuracy"], color="#2E86AB")
axes[0].set_xlabel("Accuracy de prueba")
axes[0].set_xlim(0, 1)
axes[0].set_title("Exactitud por configuración")
axes[0].grid(axis="x", alpha=0.25)

axes[1].barh(plot_data["Experimento"], plot_data["Macro F1"], color="#F18F01")
axes[1].set_xlabel("Macro F1 de prueba")
axes[1].set_xlim(0, 1)
axes[1].set_title("Macro F1 por configuración")
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
best_classification = classification_results.loc[
    classification_results["Macro F1"].idxmax()
]
worst_classification = classification_results.loc[
    classification_results["Macro F1"].idxmin()
]

print(
    f"Mejor configuración: {best_classification['Experimento']}, "
    f"accuracy={best_classification['Accuracy']:.4f} y "
    f"macro-F1={best_classification['Macro F1']:.4f}."
)
print(
    f"Menor desempeño: {worst_classification['Experimento']}, "
    f"macro-F1={worst_classification['Macro F1']:.4f}."
)
print(
    "Conclusión: una mayor separación facilita la clasificación, mientras que "
    "el ruido y la superposición entre clases dificultan el aprendizaje. Una red "
    "más grande no garantiza por sí sola una mejora; también importan la tasa de "
    "aprendizaje y la dificultad de los datos."
)

### 25.3 Experimentos de regresión

Se comparan las tres arquitecturas pedidas. El mejor modelo debe combinar **MAE y RMSE bajos** con un **\(R^2\) alto**. RMSE es más sensible que MAE a errores grandes.


In [ ]:
regression_configs = [
    ("Modelo pequeño", (16,)),
    ("Modelo mediano", (64, 32)),
    ("Modelo profundo", (128, 64, 32)),
]

regression_experiments = []
regression_predictions = {}
regression_histories = {}

for name, architecture in regression_configs:
    result, predictions, history = train_and_evaluate_regressor(name, architecture)
    regression_experiments.append(result)
    regression_predictions[name] = predictions
    regression_histories[name] = history
    print(f"Completado: {name} ({result['Épocas']} épocas)")

regression_results = pd.DataFrame(regression_experiments)
regression_results.round({"MAE": 3, "RMSE": 3, "R²": 4})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(regression_results))
width = 0.35
axes[0].bar(x - width/2, regression_results["MAE"], width, label="MAE")
axes[0].bar(x + width/2, regression_results["RMSE"], width, label="RMSE")
axes[0].set_xticks(x, regression_results["Modelo"], rotation=15)
axes[0].set_ylabel("Error")
axes[0].set_title("Errores de prueba")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(regression_results["Modelo"], regression_results["R²"], color="#4C956C")
axes[1].set_ylabel("R²")
axes[1].set_title("Coeficiente de determinación")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
best_regression = regression_results.loc[regression_results["RMSE"].idxmin()]

print(
    f"El menor RMSE lo obtuvo {best_regression['Modelo']} "
    f"({best_regression['Arquitectura']}), con "
    f"MAE={best_regression['MAE']:.3f}, "
    f"RMSE={best_regression['RMSE']:.3f} y R²={best_regression['R²']:.4f}."
)
print(
    "Conclusión: aumentar la capacidad puede ayudar hasta cierto punto, pero el "
    "resultado depende de la generalización. La arquitectura más compleja no "
    "necesariamente es la mejor en datos de prueba."
)

## 26. Preguntas de reflexión — respuestas

### Clasificación multiclase

1. **¿Por qué la capa de salida tiene cuatro neuronas?**  
   Porque existen cuatro clases y cada neurona representa la probabilidad estimada de una de ellas.

2. **¿Qué significa cada probabilidad Softmax?**  
   Indica la confianza relativa del modelo en que la observación pertenece a una clase. Las cuatro probabilidades están entre 0 y 1 y suman 1.

3. **¿Por qué se utiliza `argmax`?**  
   Porque selecciona el índice de la probabilidad más alta; ese índice corresponde a la clase predicha.

4. **¿Qué diferencia existe entre etiquetas enteras y one-hot?**  
   Una etiqueta entera representa la clase con un número, por ejemplo `2`. En one-hot se usa un vector, por ejemplo `[0, 0, 1, 0]`. Con enteros se usa normalmente `sparse_categorical_crossentropy`; con one-hot, `categorical_crossentropy`.

5. **¿Qué información aporta la matriz de confusión?**  
   Muestra cuántos ejemplos de cada clase fueron clasificados correcta o incorrectamente y permite identificar entre qué clases se producen las confusiones.

6. **¿Por qué accuracy puede ser insuficiente?**  
   Porque puede ocultar un rendimiento deficiente en clases minoritarias. En datos desbalanceados conviene revisar también precision, recall, macro-F1, balanced accuracy y la matriz de confusión.

### Regresión

7. **¿Por qué la salida utiliza activación lineal?**  
   Porque la predicción es un valor continuo y no debe quedar restringida a un intervalo como ocurriría con sigmoide o tanh.

8. **¿Qué diferencia existe entre MAE y RMSE?**  
   MAE promedia la magnitud absoluta de los errores. RMSE parte de errores al cuadrado, por lo que es más sensible a desviaciones grandes.

9. **¿Por qué RMSE penaliza más los errores grandes?**  
   Porque antes de promediar eleva cada error al cuadrado; al duplicar un error, su contribución cuadrática se multiplica por cuatro.

10. **¿Qué significa un \(R^2\) cercano a uno?**  
    Significa que el modelo explica una proporción alta de la variabilidad observada en la variable objetivo.

11. **¿Puede \(R^2\) ser negativo?**  
    Sí. Ocurre cuando el modelo predice peor que una línea base que siempre usa la media del conjunto de entrenamiento.

12. **¿Por qué MAPE debe usarse con cautela?**  
    Porque divide el error entre el valor real; si ese valor es cero o muy pequeño, el porcentaje puede ser indefinido o exageradamente grande.

13. **¿Qué información aporta el análisis de residuos?**  
    Permite detectar sesgos, patrones, valores atípicos y cambios en la dispersión del error. Una nube aleatoria alrededor de cero es una señal deseable.

14. **¿Por qué es importante comparar con una línea base?**  
    Porque permite comprobar si la red realmente aprende una relación útil. Si no supera una predicción simple, su complejidad no se justifica.


## Síntesis

En clasificación multiclase, la red transforma las características en un vector de probabilidades:

\[
\hat{\mathbf y}=[p_0,p_1,p_2,p_3],\qquad \sum_k p_k=1
\]

La clase se obtiene con `argmax`, y el desempeño debe evaluarse tanto de forma global como por clase. Los experimentos muestran que la separación y el ruido de los datos pueden influir tanto como la arquitectura.

En regresión, la salida es continua:

\[
\hat y=f(X;\theta)
\]

MAE mide el error promedio, RMSE destaca errores grandes y \(R^2\) compara la capacidad explicativa del modelo frente a la media. Aumentar capas y neuronas incrementa la cantidad de parámetros, pero no garantiza mejor generalización.

El flujo común es:

\[
\text{datos}\rightarrow\text{preparación}\rightarrow\text{modelo}\rightarrow
\text{entrenamiento}\rightarrow\text{evaluación}\rightarrow\text{interpretación}
\]


## Reto final — desarrollo

### Reto 1 y 2: clasificación multiclase desbalanceada

Se genera un problema de diez características con proporciones aproximadas de 65 %, 20 %, 10 % y 5 %. Además de accuracy se calculan macro-F1, balanced accuracy, recall por clase y matriz de confusión.


In [ ]:
X_imb, y_imb = make_classification(
    n_samples=3000,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    n_classes=4,
    n_clusters_per_class=1,
    weights=[0.65, 0.20, 0.10, 0.05],
    class_sep=1.2,
    flip_y=0.03,
    random_state=SEED,
)

X_train_full_i, X_test_i, y_train_full_i, y_test_i = train_test_split(
    X_imb, y_imb, test_size=0.20, stratify=y_imb, random_state=SEED
)
X_train_i, X_val_i, y_train_i, y_val_i = train_test_split(
    X_train_full_i, y_train_full_i, test_size=0.20,
    stratify=y_train_full_i, random_state=SEED
)
scaler_imb = StandardScaler()
X_train_i = scaler_imb.fit_transform(X_train_i)
X_val_i = scaler_imb.transform(X_val_i)
X_test_i = scaler_imb.transform(X_test_i)

imbalanced_classifier = build_classifier(
    input_dim=10, hidden_layers=(64, 32), learning_rate=0.001
)
history_imb = imbalanced_classifier.fit(
    X_train_i, y_train_i,
    validation_data=(X_val_i, y_val_i),
    epochs=220,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=18, restore_best_weights=True
    )],
    verbose=0,
)

pred_imb = np.argmax(imbalanced_classifier.predict(X_test_i, verbose=0), axis=1)
class_counts_imb = pd.Series(y_test_i).value_counts(normalize=True).sort_index()
recalls_imb = recall_score(y_test_i, pred_imb, average=None, labels=[0, 1, 2, 3])

imbalanced_metrics = pd.DataFrame({
    "Métrica": ["Accuracy", "Balanced accuracy", "Macro F1", "Accuracy de la línea base mayoritaria"],
    "Valor": [
        accuracy_score(y_test_i, pred_imb),
        balanced_accuracy_score(y_test_i, pred_imb),
        f1_score(y_test_i, pred_imb, average="macro"),
        class_counts_imb.max(),
    ],
})

print("Distribución de clases en prueba:")
print(class_counts_imb.round(4))
print("\nRecall por clase:", np.round(recalls_imb, 4))
imbalanced_metrics.round(4)

In [ ]:
cm_imb = confusion_matrix(y_test_i, pred_imb)
disp_imb = ConfusionMatrixDisplay(cm_imb, display_labels=["Clase 0", "Clase 1", "Clase 2", "Clase 3"])
disp_imb.plot(cmap="Blues")
plt.title("Reto: matriz de confusión con clases desbalanceadas")
plt.show()

print(classification_report(
    y_test_i, pred_imb,
    target_names=["Clase 0", "Clase 1", "Clase 2", "Clase 3"],
    digits=4,
))

In [ ]:
acc_imb = accuracy_score(y_test_i, pred_imb)
macro_f1_imb = f1_score(y_test_i, pred_imb, average="macro")
balanced_acc_imb = balanced_accuracy_score(y_test_i, pred_imb)

print(
    f"El modelo obtuvo accuracy={acc_imb:.4f}, balanced accuracy={balanced_acc_imb:.4f} "
    f"y macro-F1={macro_f1_imb:.4f}."
)
print(
    "Conclusión: accuracy no es suficiente en este conjunto. La clase mayoritaria "
    "puede dominar el resultado global, mientras que macro-F1, balanced accuracy, "
    "recall por clase y la matriz de confusión muestran si las clases minoritarias "
    "también se reconocen correctamente."
)

### Reto 3 y 4: regresor con L2 y Dropout

Para aislar el efecto de la regularización, ambos modelos usan la misma arquitectura `10 → 64 → 32 → 16 → 1`. El modelo regularizado agrega L2 de 0.001 y Dropout de 0.20 después de cada capa oculta.


In [ ]:
challenge_results = []
challenge_predictions = {}

original_result, original_predictions, original_history = train_and_evaluate_regressor(
    "Original", (64, 32, 16), l2_value=0.0, dropout_rate=0.0
)
regularized_result, regularized_predictions, regularized_history = train_and_evaluate_regressor(
    "L2 + Dropout", (64, 32, 16), l2_value=0.001, dropout_rate=0.20
)

challenge_results.extend([original_result, regularized_result])
challenge_predictions["Original"] = original_predictions
challenge_predictions["L2 + Dropout"] = regularized_predictions

regularization_comparison = pd.DataFrame(challenge_results)
regularization_comparison.round({"MAE": 3, "RMSE": 3, "R²": 4})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (name, predictions) in zip(axes, challenge_predictions.items()):
    residual_values = y_test_r - predictions
    ax.scatter(predictions, residual_values, alpha=0.70)
    ax.axhline(0, color="black", linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Valor predicho")
    ax.set_ylabel("Residuo")
    ax.grid(alpha=0.25)

fig.suptitle("Comparación de residuos: modelo original y regularizado")
plt.tight_layout()
plt.show()

In [ ]:
best_challenge = regularization_comparison.loc[
    regularization_comparison["RMSE"].idxmin()
]

print(
    f"En esta ejecución, el menor RMSE fue el de {best_challenge['Modelo']}: "
    f"MAE={best_challenge['MAE']:.3f}, RMSE={best_challenge['RMSE']:.3f} "
    f"y R²={best_challenge['R²']:.4f}."
)
print(
    "L2 limita pesos excesivamente grandes y Dropout reduce la dependencia entre "
    "neuronas. Su utilidad se confirma si mejoran las métricas de prueba o producen "
    "residuos más estables; no se debe asumir que siempre mejorarán el resultado."
)

## Conclusión general de la entrega

Las actividades muestran que el desempeño de una red neuronal no depende únicamente de hacerla más grande. En clasificación influyen la arquitectura, la tasa de aprendizaje y, de manera muy marcada, la calidad y separación de los datos. En problemas desbalanceados es obligatorio complementar accuracy con métricas por clase.

En regresión, la comparación debe considerar simultáneamente MAE, RMSE, \(R^2\), la línea base y los residuos. La regularización puede favorecer la generalización, pero su efecto debe verificarse experimentalmente. Las tablas y gráficas anteriores se calculan directamente durante la ejecución del notebook y permiten sustentar estas conclusiones con evidencia.
